In [4]:
# install pacakges
!pip install -q huggingface_hub pyarrow tqdm

In [5]:
# imports
from huggingface_hub import hf_hub_download

from pathlib import Path
import zipfile
import json
import hashlib

import pyarrow as pa
import pyarrow.parquet as pq

from tqdm import tqdm


In [23]:
# data paths
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

TRAIN_DIR = DATA_DIR / "Win32_train"
TEST_DIR = DATA_DIR / "Win32_test"

TRAIN_DIR.mkdir(exist_ok=True)
TEST_DIR.mkdir(exist_ok=True)


DETECTION_PARQUET = DATA_DIR / "win32_detection_train_20pct.parquet"
BEHAVIOR_PARQUET = DATA_DIR / "win32_behavior_train_20pct.parquet"
TEST_PARQUET = DATA_DIR / "win32_test.parquet"

In [8]:
# download zip iles
train_zip = hf_hub_download(
    repo_id="joyce8/EMBER2024",
    repo_type="dataset",
    filename="Win32_train.zip"
)

In [9]:
test_zip = hf_hub_download(
    repo_id="joyce8/EMBER2024",
    repo_type="dataset",
    filename="Win32_test.zip"
)

In [10]:
print(f"Train: {train_zip}")
print(f"Test: {test_zip}")

Train: /Users/sophieliu/.cache/huggingface/hub/datasets--joyce8--EMBER2024/snapshots/3d23efef7c0f0b702c5024400cfff4c3744a3832/Win32_train.zip
Test: /Users/sophieliu/.cache/huggingface/hub/datasets--joyce8--EMBER2024/snapshots/3d23efef7c0f0b702c5024400cfff4c3744a3832/Win32_test.zip


In [ ]:
with zipfile.ZipFile(train_zip, "r") as z:
    z.extractall(TRAIN_DIR)

with zipfile.ZipFile(test_zip, "r") as z:
    z.extractall(TEST_DIR)

Train files: 0
Test files: 0


In [14]:
# jsonl reader
def json_stream(folder):
    for file in folder.glob("*.jsonl"):
        with open(file, "r") as f:
            for line in f:
                yield json.loads(line)

In [15]:
print("Train files:", len(list(TRAIN_DIR.glob("*.jsonl"))))
print("Test files:", len(list(TEST_DIR.glob("*.jsonl"))))

Train files: 52
Test files: 12


In [19]:
KEEP_PERCENT = 0.20
SEED = 42


DETECTION_FEATURES = [
    "sha256",
    "label",
    "general",
    "histogram",
    "byteentropy",
    "strings",
    "imports"
]


BEHAVIOR_FEATURES = [
    "sha256",
    "label",
    "family",
    "mbc",
    "ttps",
    "behavior"
]

In [30]:
# function to get WIN32 subset
def keep_sample(row, percent=0.20):
    # hash sha256 ID
    h = hashlib.sha256(
        (row["sha256"] + str(SEED)).encode()
    ).hexdigest()

    normalized = int(h[:8], 16) / 0xffffffff

    return normalized < percent

# row cleaning
def clean_detection_row(row):
    return {
        "sha256": row.get("sha256"),
        "label": row.get("label"),

        # convert nested structures to stable strings
        "general": json.dumps(row.get("general", {})),
        "strings": json.dumps(row.get("strings", {})),
        "imports": json.dumps(row.get("imports", {}))
    }

def clean_behavior_row(row):
    return {
        "sha256": row.get("sha256"),
        "label": row.get("label"),
        "family": row.get("family"),
        "behavior": row.get("behavior") or [],
        
        "mbc": json.dumps(row.get("mbc", [])),
        "ttps": json.dumps(row.get("ttps", []))
        
    }

In [ ]:
# for malware classification
def save_parquet(input_dir, output_file, cleaner):
    writer = None
    count = 0 # number of saved samples
    seen = set()

    stream = json_stream(input_dir)
    for row in tqdm(stream, desc="Processing Samples"):
        # skip non-WIN32 subset samples
        if not keep_sample(row):
            continue

        cleaned = cleaner(row)
        sha = cleaned["sha256"]
        if sha in seen:
            continue
        seen.add(sha)
        
        table = pa.Table.from_pylist([cleaned])
        if writer is None:
            writer = pq.ParquetWriter(
                output_file,
                table.schema,
                compression="snappy"
            )
        # append sample to parquet file
        writer.write_table(table)
        count += 1

    if writer:
        writer.close()
    print(f"Saved {count:,} detection samples")

In [36]:
# save files to parquet
save_parquet(
    TRAIN_DIR,
    DETECTION_PARQUET,
    clean_detection_row
)
# save_parquet(
#     TRAIN_DIR,
#     BEHAVIOR_PARQUET,
#     clean_behavior_row
# )

3120000it [07:34, 6858.79it/s]


Saved 624,250 detection samples


In [40]:
import pandas as pd
df = pd.read_parquet(DATA_DIR / "win32_detection_train_20pct.parquet")

print("rows:", len(df))
print("duplicate rows:", df.duplicated().sum())
print("duplicate sha256:", df["sha256"].duplicated().sum())


rows: 624250
duplicate rows: 312125
duplicate sha256: 312125


In [44]:
import os

if os.path.exists(DETECTION_PARQUET):
    os.remove(DETECTION_PARQUET)

In [42]:
import pandas as pd

df = pd.read_parquet(DETECTION_PARQUET)

print(len(df))
print(df["sha256"].nunique())

624250
312125


In [43]:
df = df.drop_duplicates("sha256")

print(len(df))

312125
